In [1]:
import argparse
from pathlib import Path
import numpy as np
import pyarrow.parquet as pq
import pyarrow as pa
import pandas as pd
from tqdm import tqdm

import pybacktest
from pybacktest.bookcore import BookCore



In [2]:
data_dir = Path(r'E:\tmp\OKX-Books-BTC-USDT-400')
pfs = list(data_dir.glob("*.parquet"))

In [3]:
df_back = pq.ParquetFile(pfs[0]).read().to_pandas()

In [9]:
a = list(df_back.itertuples())

In [ ]:
a[0]._asdict()

{'Index': 0,
 'arg': {'channel': 'books', 'instId': 'BTC-USDT'},
 'data': {'asks': array([array(['83404', '0', '0', '0'], dtype=object),
         array(['83404.3', '0.1189398', '0', '3'], dtype=object),
         array(['83406', '0', '0', '0'], dtype=object),
         array(['83408.5', '0', '0', '0'], dtype=object),
         array(['83409.9', '0.03320099', '0', '1'], dtype=object),
         array(['83410.3', '0.05325002', '0', '1'], dtype=object),
         array(['83412.8', '0.09736682', '0', '2'], dtype=object),
         array(['83414.3', '0', '0', '0'], dtype=object),
         array(['83414.7', '0.18420673', '0', '1'], dtype=object),
         array(['83415.3', '0', '0', '0'], dtype=object),
         array(['83415.4', '0.0383506', '0', '1'], dtype=object),
         array(['83416', '0.02249397', '0', '1'], dtype=object),
         array(['83416.3', '0.03287194', '0', '1'], dtype=object),
         array(['83418.8', '0', '0', '0'], dtype=object),
         array(['83418.9', '0.00890882', '0

In [16]:
df = df_back.copy()
tmp = df[df['action'] == 'snapshot'][['ts', 'action']]

In [37]:
tss = np.append(tmp.index.values, df.index.values[-1])
all(tss[i] + 100_000 >=tss[i + 1] for i in range(len(tss) - 1))

True

In [48]:
table = pq.read_table(pfs[0])

In [49]:
tmp = table.to_pylist()

In [9]:
interval = 100000
bc = None
iter_rows = 0
df = df_back.copy()
# dps = list(map(convert, df.to_dict(orient='records')))
dps = df.to_dict(orient='records')
for dp in dps:
    if bc == None:
        if dp['action'] != 'snapshot':
            continue
        instId = dp['arg']['instId']
        bc = BookCore(instId)
    
    try: 
        bc.set_datapoint(dp)
    except Exception as e:
        print(f"Error processing datapoint: {e}")
        print(f"Datapoint: {dp}")
        raise e
    iter_rows += 1

    if iter_rows >= interval: # should insert a snapshot
        asks_bl = [[str(bl.price),str(bl.amount),'0',str(bl.count)] for bl in bc.asks]
        bids_bl = [[str(bl.price),str(bl.amount),'0',str(bl.count)] for bl in bc.bids]
        # update the row with the new snapshot data
        dp['data']['asks'] = asks_bl
        dp['data']['bids'] = bids_bl
        dp['action'] = 'snapshot'
        iter_rows = 0
        break
table = pa.Table.from_pylist(dps)

In [7]:
def convert(datapoint):
    new_data = {
        'asks': [bl.tolist() for bl in datapoint['data']['asks']],
        'bids': [bl.tolist() for bl in datapoint['data']['bids']],
        'checksum': datapoint['data']['checksum'],
        'prevSeqId': datapoint['data']['prevSeqId'],
        'seqId': datapoint['data']['seqId'],
        'ts': datapoint['data']['ts']
    }
    return {
        'arg': datapoint['arg'],
        'data': new_data,
        'action': datapoint['action'],
        'ts': datapoint['ts']
    }
list(map(convert, df.to_dict('records')))[:10]

KeyboardInterrupt: 

In [ ]:
df_back[df_back['action'] == 'snapshot'].index

Index([233285, 391568, 657489, 821565, 843269, 848913, 852340, 951367], dtype='int64')